# Chapter 1: Data - Heuristic Clustering & Blockchain Data Extraction

This notebook queries raw blockchain data, applies heuristic clustering, formats transaction flows, and stores the extracted data into a local MySQL database. This data forms the foundational dataset for time zone analysis.

**Methodology:**
- Reads an initial dataset of categorized Bitcoin addresses.
- Configures rotating proxies and cached sessions to prevent rate-limiting and redundant network calls.
- Employs memory-efficient generator functions and Pandas pre-filtering to prevent Out-Of-Memory crashes.
- Commits data to a local MySQL database iteratively in chunks to maintain stability and prevent connection leaks.

In [ ]:
import requests
import requests_cache
from heuristic_clustering import HeuristicClustering
from get_address_info import GetAddressInfo
import datetime
import pymysql
import pandas as pd
import os
from dotenv import load_dotenv

### 1. Data Loading
Read the initial dataset containing categorized Bitcoin addresses and their associated entities. Dataset was extracted from the local MySQL server.

In [ ]:
data = pd.read_csv("test.csv")
data.head()

### 2. Database & Chunking Configuration
Define memory-efficient generator functions to chunk large transaction arrays, preventing duplication in RAM. Securely loads credentials and writes data batches to the local MySQL database while strictly managing connection states.

In [ ]:
def get_chunks(data_list, chunk_size):
    """
    Yield successive chunks from a list.
    Using a generator (yield) instead of list slicing prevents duplicate copies 
    of massive transaction arrays in memory, mitigating OOM crashes.
    """
    for i in range(0, len(data_list), chunk_size):
        yield data_list[i:i + chunk_size]

def data_write(chunk_size, batch_blockchain_data, batch_tx_inputs, batch_tx_outputs, batch_entities_heur):
    conn = pymysql.connect(
        host='127.0.0.1',
        user='root',
        password=os.getenv("PASSWORD_MY_SQL"),
        database='data'
    )
    
    try:
        with conn.cursor() as cursor:
            # Write Blockchain Data
            sql_bc = "INSERT IGNORE INTO blockchain_data (txid, num_inputs, num_outputs, fee, mempool_entry_time, block_height) VALUES (%s, %s, %s, %s, %s, %s)"
            for chunk in get_chunks(batch_blockchain_data, chunk_size):
                cursor.executemany(sql_bc, chunk)
                conn.commit()

            # Write TX Inputs
            sql_in = "INSERT IGNORE INTO tx_inputs (txid, input_order, address, value) VALUES (%s, %s, %s, %s)"
            for chunk in get_chunks(batch_tx_inputs, chunk_size):
                cursor.executemany(sql_in, chunk)
                conn.commit()

            # Write TX Outputs
            sql_out = "INSERT IGNORE INTO tx_outputs (txid, output_order, address, value) VALUES (%s, %s, %s, %s)"
            for chunk in get_chunks(batch_tx_outputs, chunk_size):
                cursor.executemany(sql_out, chunk)
                conn.commit()
            
            # Write Heuristics
            sql_heur = "INSERT IGNORE INTO entities_heur (entity, address) VALUES (%s, %s)"
            for chunk in get_chunks(batch_entities_heur, chunk_size):
                cursor.executemany(sql_heur, chunk)
                conn.commit()
    finally:
        conn.close() 


### 3. Proxy & Cache Configuration
Securely load a list of 100 Webshare proxies to distribute HTTP/HTTPS requests and set up a 30-day cache expiration to handle network redundancy gracefully.

In [ ]:
# Load environment variables (ensure your .env file is configured correctly)
load_dotenv()

path = os.getenv("RAW_DATA_PATH")
proxies_dict_list = []

# Open the text file containing the proxies in read mode with UTF-8 encoding
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # split the TXT format IP:PORT:USER:PASS
        ip, port, user, pwd = line.split(":")
        
        # create a dict directly for requests
        proxy_dict = {
            "http": f"http://{user}:{pwd}@{ip}:{port}",
            "https": f"http://{user}:{pwd}@{ip}:{port}"
        }
        proxies_dict_list.append(proxy_dict)

session = requests_cache.CachedSession('api_cache', expire_after=datetime.timedelta(days=30))

### 4. Main Clustering Execution
The primary data processing loop. Queries the necessary blockchain data using heuristic clustering, and pushes the consolidated lists to the database.

In [ ]:
a = 0
start_point = 1000

# Pre-filter targets using .isin() to avoid iterating through unnecessary rows
targets = data

for index, row in targets.iterrows():
    if index > start_point:
        print(index)
        address = row["address"]
        entity = row["entity"]
        
        heur = HeuristicClustering(address, session, a, proxies_dict_list)
        heur.heuristic_clus()
        a = heur.a
        
        addresses_heur = [[entity, addr] for addr in heur.old_addresses]

        data_write(1000, heur.blockchain_final_data, heur.inputs_final_data, heur.outputs_final_data, addresses_heur)
        print(f"Address {address} written with {len(heur.old_addresses)} linked addresses.")